In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

params = pd.read_csv('../team_parameters.csv')
cws_meta = params[params['team_code'] == 'CWS'].iloc[0]

cws = pd.read_csv(f"../../data/{cws_meta['dataset_file']}")
cws = cws[(cws['season'] >= int(cws_meta['data_start_year'])) & (cws['season'] <= int(cws_meta['data_end_year']))].copy()
cws = cws.dropna(subset=['start_hour', 'total_runs', 'home_runs_scored', 'away_runs_scored', 'temp_f', 'rhum', 'pres'])

cws['game_date'] = pd.to_datetime(cws['game_date'])
cws['month'] = cws['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
cws['month_name'] = cws['month'].map(month_map)
cws['time_of_day'] = np.where(cws['start_hour'] < 17, 'Day', 'Night')

cws['temp_bin'] = pd.qcut(cws['temp_f'], q=5, duplicates='drop')
cws['rhum_bin'] = pd.qcut(cws['rhum'], q=5, duplicates='drop')
cws['pres_bin'] = pd.qcut(cws['pres'], q=5, duplicates='drop')

print(f"Dataset: {cws_meta['dataset_file']} ({int(cws_meta['data_start_year'])}-{int(cws_meta['data_end_year'])})")
print(f'Total White Sox home games with complete day/night + weather data: {len(cws)}')
print('\nDay/Night game counts:')
print(cws['time_of_day'].value_counts())

In [ ]:
# ============================================================
# Day vs Night Summary Table
# ============================================================
day_night_summary = cws.groupby('time_of_day', observed=True).agg(
    games=('game_pk', 'size'),
    total_runs_mean=('total_runs', 'mean'),
    home_runs_mean=('home_runs_scored', 'mean'),
    away_runs_mean=('away_runs_scored', 'mean'),
    hits_mean=('hits', 'mean'),
    home_runs_hit_mean=('home_runs_hit', 'mean'),
    walks_mean=('walks', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
    temp_mean=('temp_f', 'mean'),
    rhum_mean=('rhum', 'mean'),
    pres_mean=('pres', 'mean'),
    wind_mean=('wspd_mph', 'mean')
).round(3)

day_night_summary.index.name = 'Game Type'
day_night_summary.columns = [
    'Games', 'Total Runs Mean', 'Home Runs Mean', 'Away Runs Mean',
    'Hits Mean', 'Home Runs Hit Mean', 'Walks Mean', 'Strikeouts Mean',
    'Temp Mean', 'Humidity Mean', 'Pressure Mean', 'Wind Mean'
]
day_night_summary

In [ ]:
# ============================================================
# Main Day/Night Diagnostic Figure
# ============================================================
order = ['Day', 'Night']
x = np.arange(len(order))
summary = day_night_summary.loc[order].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

axes[0, 0].bar(x - 0.25, summary['Total Runs Mean'], width=0.25, color='#d62728', label='Total runs')
axes[0, 0].bar(x, summary['Home Runs Mean'], width=0.25, color='#1f77b4', label='Home runs')
axes[0, 0].bar(x + 0.25, summary['Away Runs Mean'], width=0.25, color='#ff7f0e', label='Away runs')
axes[0, 0].set_title('Run Production by Time of Day')
axes[0, 0].set_ylabel('Runs')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(order)
axes[0, 0].legend()

axes[0, 1].boxplot([cws.loc[cws['time_of_day'] == label, 'total_runs'].dropna().values for label in order], labels=order, patch_artist=True,
                   boxprops=dict(facecolor='#f2a5a0', alpha=0.7), medianprops=dict(color='black', linewidth=2))
axes[0, 1].set_title('Total Runs Distribution: Day vs Night')
axes[0, 1].set_ylabel('Total runs')

axes[1, 0].bar(x - 0.2, summary['Temp Mean'], width=0.2, color='#ff7f0e', label='Temperature')
axes[1, 0].bar(x, summary['Humidity Mean'], width=0.2, color='#17becf', label='Humidity')
axes[1, 0].bar(x + 0.2, summary['Wind Mean'], width=0.2, color='#8c564b', label='Wind')
axes[1, 0].set_title('Weather Differences by Time of Day')
axes[1, 0].set_ylabel('Temperature / Humidity / Wind')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(order)
axes[1, 0].legend()

axes[1, 1].boxplot([cws.loc[cws['time_of_day'] == label, 'pres'].dropna().values for label in order], labels=order, patch_artist=True,
                   boxprops=dict(facecolor='#b7b7b7', alpha=0.8), medianprops=dict(color='black', linewidth=2))
axes[1, 1].set_title('Pressure Distribution: Day vs Night')
axes[1, 1].set_ylabel('Pressure (hPa)')

for ax in axes.flat:
    ax.grid(alpha=0.25)

plt.suptitle('Guaranteed Rate Field: Is the Night Run Boost Mainly a Weather Story?', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Monthly Pattern: Are Night Games Concentrated in Different Parts of the Season?
# ============================================================
monthly_summary = cws.groupby(['month_name', 'time_of_day'], observed=True).agg(
    games=('game_pk', 'size'),
    total_runs_mean=('total_runs', 'mean'),
    temp_mean=('temp_f', 'mean'),
    rhum_mean=('rhum', 'mean'),
    pres_mean=('pres', 'mean')
).reset_index()

month_order = [m for m in ['Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct'] if m in monthly_summary['month_name'].unique()]
monthly_pivot_runs = monthly_summary.pivot(index='month_name', columns='time_of_day', values='total_runs_mean').reindex(month_order)
monthly_pivot_temp = monthly_summary.pivot(index='month_name', columns='time_of_day', values='temp_mean').reindex(month_order)
monthly_pivot_humidity = monthly_summary.pivot(index='month_name', columns='time_of_day', values='rhum_mean').reindex(month_order)
monthly_pivot_pressure = monthly_summary.pivot(index='month_name', columns='time_of_day', values='pres_mean').reindex(month_order)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

monthly_pivot_runs.plot(kind='bar', ax=axes[0, 0], color=['#4c78a8', '#e45756'])
axes[0, 0].set_title('Monthly Total Runs: Day vs Night')
axes[0, 0].set_ylabel('Total runs')
axes[0, 0].legend(title='Game Type')

monthly_pivot_temp.plot(kind='bar', ax=axes[0, 1], color=['#f28e2b', '#edc948'])
axes[0, 1].set_title('Monthly Temperature: Day vs Night')
axes[0, 1].set_ylabel('Temperature (degF)')
axes[0, 1].legend(title='Game Type')

monthly_pivot_humidity.plot(kind='bar', ax=axes[1, 0], color=['#76b7b2', '#59a14f'])
axes[1, 0].set_title('Monthly Humidity: Day vs Night')
axes[1, 0].set_ylabel('Humidity (%)')
axes[1, 0].legend(title='Game Type')

monthly_pivot_pressure.plot(kind='bar', ax=axes[1, 1], color=['#bab0ab', '#9c755f'])
axes[1, 1].set_title('Monthly Pressure: Day vs Night')
axes[1, 1].set_ylabel('Pressure (hPa)')
axes[1, 1].legend(title='Game Type')

for ax in axes.flat:
    ax.grid(axis='y', alpha=0.25)

plt.suptitle('Seasonal Composition Check: Do Night Games Live in Different Weather Windows?', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

monthly_summary

In [ ]:
# ============================================================
# Conditional Run Comparisons Within Temperature / Humidity / Pressure Buckets
# ============================================================
def conditional_day_night(frame, bin_col, label_name):
    grouped = frame.groupby([bin_col, 'time_of_day'], observed=True).agg(
        games=('game_pk', 'size'),
        total_runs_mean=('total_runs', 'mean'),
        home_runs_mean=('home_runs_scored', 'mean'),
        away_runs_mean=('away_runs_scored', 'mean')
    ).reset_index()
    grouped[label_name] = grouped[bin_col].astype(str)
    pivot_runs = grouped.pivot(index=label_name, columns='time_of_day', values='total_runs_mean')
    pivot_games = grouped.pivot(index=label_name, columns='time_of_day', values='games')
    pivot_runs['Night_minus_Day'] = pivot_runs.get('Night', np.nan) - pivot_runs.get('Day', np.nan)
    return grouped, pivot_runs, pivot_games

temp_grouped, temp_pivot, temp_games = conditional_day_night(cws, 'temp_bin', 'temp_label')
rhum_grouped, rhum_pivot, rhum_games = conditional_day_night(cws, 'rhum_bin', 'rhum_label')
pres_grouped, pres_pivot, pres_games = conditional_day_night(cws, 'pres_bin', 'pres_label')

display(temp_pivot)
display(rhum_pivot)
display(pres_pivot)

In [ ]:
# ============================================================
# Visualizing Whether the Night Effect Survives Inside Weather Buckets
# ============================================================
fig, axes = plt.subplots(3, 2, figsize=(18, 15))

temp_pivot[['Day', 'Night']].plot(kind='bar', ax=axes[0, 0], color=['#4c78a8', '#e45756'])
axes[0, 0].set_title('Total Runs by Temperature Quintile')
axes[0, 0].set_ylabel('Total runs')
axes[0, 0].legend(title='Game Type')

temp_pivot['Night_minus_Day'].plot(kind='bar', ax=axes[0, 1], color='#9467bd')
axes[0, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[0, 1].set_title('Night Minus Day: Temperature Quintiles')
axes[0, 1].set_ylabel('Run difference')

rhum_pivot[['Day', 'Night']].plot(kind='bar', ax=axes[1, 0], color=['#4c78a8', '#e45756'])
axes[1, 0].set_title('Total Runs by Humidity Quintile')
axes[1, 0].set_ylabel('Total runs')
axes[1, 0].legend(title='Game Type')

rhum_pivot['Night_minus_Day'].plot(kind='bar', ax=axes[1, 1], color='#2ca02c')
axes[1, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[1, 1].set_title('Night Minus Day: Humidity Quintiles')
axes[1, 1].set_ylabel('Run difference')

pres_pivot[['Day', 'Night']].plot(kind='bar', ax=axes[2, 0], color=['#4c78a8', '#e45756'])
axes[2, 0].set_title('Total Runs by Pressure Quintile')
axes[2, 0].set_ylabel('Total runs')
axes[2, 0].legend(title='Game Type')

pres_pivot['Night_minus_Day'].plot(kind='bar', ax=axes[2, 1], color='#8c564b')
axes[2, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[2, 1].set_title('Night Minus Day: Pressure Quintiles')
axes[2, 1].set_ylabel('Run difference')

for ax in axes.flat:
    ax.grid(axis='y', alpha=0.25)

plt.suptitle('Conditional Check: Does the White Sox Night-Run Edge Persist Within Similar Weather?', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()